In [16]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated,Literal
from langchain_core.messages import BaseMessage,HumanMessage
from pydantic import BaseModel,Field
import operator
from langchain_core.prompts import ChatPromptTemplate
import os
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
load_dotenv()

True

In [4]:
class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

model=ChatGoogleGenerativeAI(model="gemini-2.5-flash",api_key=os.getenv("GEMINI_API_KEY"))


In [5]:
def chat_node(state:ChatState):
    messages=state["messages"]
    response=model.invoke(messages)
    return {"messages":[response]}

In [17]:
graph=StateGraph(ChatState)
checkpoint=MemorySaver()
graph.add_node("chat_node",chat_node)
graph.add_edge(START,"chat_node")
graph.add_edge("chat_node",END)
chatbot=graph.compile(checkpointer=checkpoint)

In [10]:
istate={
    'messages':[HumanMessage(content="what is the capital of india?")]
}
result=chatbot.invoke(istate)["messages"]

In [11]:
result

[HumanMessage(content='what is the capital of india?', additional_kwargs={}, response_metadata={}, id='9f6e483b-394c-4627-93c9-8f30931a8f21'),
 AIMessage(content='The capital of India is **New Delhi**.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d303a-8c63-7530-b10a-d9931a138ff6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 24, 'total_tokens': 32, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 15}})]

In [14]:
result[-1].content

'The capital of India is **New Delhi**.'

In [18]:

thread_id='1'
while True:
    user_message=input("Type here :")
    print('User:',user_message)
    if user_message.strip().lower() in ['exit','bye','quit']:
        break
    config={'configurable':{'thread_id':thread_id}}
    response=chatbot.invoke({'messages':[HumanMessage(content=user_message)]},config=config)
    print('AI:',response['messages'][-1].content)
    



User: hi my name is nitish
AI: Hi Nitish! Nice to meet you.

How can I help you today?
User: whats my name
AI: Your name is Nitish.
User: exit
